# Reshaping and Tidy Data

## Overview

**Tidy data** (Wickham 2014) has three properties: each variable is a column, each observation is a row, each value is a cell. Most analysis functions expect tidy (long) format; most spreadsheet data arrives in wide format.

| Format | Description | Common source | R equivalent |
|---|---|---|---|
| **Wide** | One row per unit; repeated measures as columns | Spreadsheets, survey exports | `tidyr::pivot_wider` |
| **Long (tidy)** | One row per observation; variable name in its own column | Databases, analysis-ready | `tidyr::pivot_longer` |

**Key pandas functions:**
- `melt()` / `pd.melt()` — wide → long
- `pivot()` / `pivot_table()` — long → wide
- `stack()` / `unstack()` — MultiIndex reshaping
- `explode()` — list-valued cells → rows

---

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Wide format: each species as its own column (common from field data entry)
n_sites = 30
species = ['sp_A', 'sp_B', 'sp_C', 'sp_D', 'sp_E']

wide = pd.DataFrame({
    'site_id':   [f'SITE_{i:03d}' for i in range(1, n_sites+1)],
    'catchment': rng.choice(['North', 'South', 'East'], n_sites),
    'treatment': rng.choice(['control', 'restored'], n_sites),
    **{sp: rng.integers(0, 20, n_sites) for sp in species}
})

print('Wide format:')
print(wide.head())

---
## Wide → Long: `melt()`

In [ ]:
# melt(): id_vars stay fixed; value_vars become rows
long = wide.melt(
    id_vars    = ['site_id', 'catchment', 'treatment'],
    value_vars = species,
    var_name   = 'species',
    value_name = 'abundance'
)

print(f'Wide shape: {wide.shape} → Long shape: {long.shape}')
print(f'(30 sites × 5 species = {30*5} rows in long format)\n')
print(long.head(8))

# Now easy to groupby and summarise across species
total_by_site = (
    long.groupby('site_id')['abundance']
    .sum()
    .reset_index()
    .rename(columns={'abundance': 'total_abundance'})
)
print(total_by_site.head())

---
## Long → Wide: `pivot()` and `pivot_table()`

In [ ]:
# pivot(): requires unique (index, columns) combinations
wide_recovered = long.pivot(
    index   = 'site_id',
    columns = 'species',
    values  = 'abundance'
).reset_index()
wide_recovered.columns.name = None   # remove the 'species' label on columns axis
print('Pivoted back to wide:')
print(wide_recovered.head(3))

# pivot_table(): handles duplicate (index, columns) combinations via aggregation
# Useful when multiple records exist per site-species combination
species_by_catchment = pd.pivot_table(
    long,
    values  = 'abundance',
    index   = 'catchment',
    columns = 'species',
    aggfunc = 'mean',
    fill_value = 0
).round(1)
print('\nMean abundance by catchment and species:')
print(species_by_catchment)

---
## Multiple Measurements Per Site: Repeated Measures Reshaping

In [ ]:
# Simulate: same sites measured at 3 time points, 2 nutrients
n_sites = 20
timepoints = ['t1', 't2', 't3']
nutrients = ['nitrate', 'phosphorus']

wide2 = pd.DataFrame({'site_id': [f'SITE_{i:03d}' for i in range(1, n_sites+1)]})
for t in timepoints:
    for nut in nutrients:
        wide2[f'{nut}_{t}'] = rng.gamma(2, 2, n_sites).round(2)

print('Wide (multiple variables × timepoints):')
print(wide2.head(3))

# melt then split the combined column name
long2 = wide2.melt(id_vars='site_id', var_name='variable', value_name='value')
long2[['nutrient', 'timepoint']] = long2['variable'].str.rsplit('_', n=1, expand=True)
long2 = long2.drop(columns='variable')

print('\nLong with separate nutrient and timepoint columns:')
print(long2.head(8))

---
## Exploding List-Valued Cells

In [ ]:
# Common when data arrives from JSON or web APIs
nested = pd.DataFrame({
    'site_id':  ['SITE_001', 'SITE_002', 'SITE_003'],
    'species_observed': [['sp_A', 'sp_B', 'sp_C'], ['sp_B'], ['sp_A', 'sp_D']],
    'richness': [3, 1, 2]
})
print('Nested:')
print(nested)

exploded = nested.explode('species_observed').reset_index(drop=True)
print('\nExploded:')
print(exploded)

---

## Common Pitfalls

**1. Using `pivot()` when there are duplicate index-column combinations**  
`pivot()` raises a `ValueError` if the same (index, column) pair appears more than once — it cannot know which value to use. Use `pivot_table()` with an explicit `aggfunc` whenever rows might not be unique per combination. If duplicates are unexpected, investigate their origin before aggregating.

**2. Losing metadata columns during `melt()`**  
Any column not listed in `id_vars` is treated as a value column and melted into rows. If you have 20 columns and forget to list 2 ID columns in `id_vars`, they silently become value rows. Always explicitly list every identifier column; do not rely on positional defaults.

**3. Not removing the columns axis name after `pivot()`**  
After pivoting, the columns axis retains the name of the original variable column (e.g. `species`). This causes unexpected behaviour when merging or renaming. Always set `df.columns.name = None` after pivoting to produce a clean column index.

**4. Splitting compound column names with fixed separators**  
Using `str.split('_', expand=True)` on `'nitrate_conc_t1'` produces three parts, not two. Use `str.rsplit('_', n=1)` to split on the last underscore only, or use a more specific pattern. Always inspect the resulting split on a sample of values before applying to the full column.

**5. Confusing `stack()` / `unstack()` with `melt()` / `pivot()`**  
`stack()` and `unstack()` operate on the MultiIndex and are appropriate for hierarchically indexed DataFrames. `melt()` and `pivot()` operate on regular columns. Using `stack()` on a DataFrame without a MultiIndex or with mixed column types produces unexpected results. For most reshaping tasks on regular DataFrames, `melt()` and `pivot_table()` are clearer and safer.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*